**Computing Yearly Fleet Shares**

In [1]:
import json
import pandas as pd
from collections import Counter

JSON_FILE = r"C:\Users\mogul\OneDrive\Masaüstü\mt_emre\digital_twin\simulation\year_statistic.json"

with open(JSON_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

totals = Counter()
total_vehicles = 0

for _, entry in data.items():
    total_vehicles += entry.get("vehicleCount", 0)
    for veh_type, count in entry.get("vehicleTypeCount", {}).items():
        totals[veh_type] += count

dist = pd.DataFrame(
    [{"vehicle_type_json": k, "count": v} for k, v in totals.items()]
).sort_values("count", ascending=False)

dist["share"] = dist["count"] / dist["count"].sum()

print("Total vehicles:", total_vehicles)
print(dist)

Total vehicles: 274674075
   vehicle_type_json      count     share
4                CAR  237610461  0.865063
2                VAN   22469475  0.081804
6              TRUCK    5371064  0.019554
5          MOTORBIKE    3063696  0.011154
0                BUS    2184970  0.007955
7      TRUCK_TRAILER    1661122  0.006048
1        CAR_TRAILER    1281726  0.004666
3  TRUCK_SEMITRAILER    1031561  0.003756


**MAP JSON types to SUMO ids**

In [2]:
mapping = {
    "CAR": "car",
    "VAN": "van",
    "TRUCK": "truck",
    "BUS": "BUS",
    "MOTORBIKE": "motorcycle",
    "CAR_TRAILER": "car_trailer",
    "TRUCK_TRAILER": "truck_trailer",
    "TRUCK_SEMITRAILER": "semitrailer",
}

dist["sumo_vtype"] = dist["vehicle_type_json"].map(mapping)

sumo_dist = (
    dist.dropna(subset=["sumo_vtype"])
        .groupby("sumo_vtype", as_index=False)["count"]
        .sum()
)

sumo_dist["probability"] = sumo_dist["count"] / sumo_dist["count"].sum()

print(sumo_dist.sort_values("probability", ascending=False))

      sumo_vtype      count  probability
1            car  237610461     0.865063
7            van   22469475     0.081804
5          truck    5371064     0.019554
3     motorcycle    3063696     0.011154
0            BUS    2184970     0.007955
6  truck_trailer    1661122     0.006048
2    car_trailer    1281726     0.004666
4    semitrailer    1031561     0.003756


**Create a vTypeDistribution**

In [3]:
lines = []
lines.append('<vTypeDistribution id="trafficMix">')

for _, row in sumo_dist.sort_values("probability", ascending=False).iterrows():
    lines.append(
        f'    <vType id="{row["sumo_vtype"]}" probability="{row["probability"]:.6f}"/>'
    )

lines.append("</vTypeDistribution>")

print("\n".join(lines))

<vTypeDistribution id="trafficMix">
    <vType id="car" probability="0.865063"/>
    <vType id="van" probability="0.081804"/>
    <vType id="truck" probability="0.019554"/>
    <vType id="motorcycle" probability="0.011154"/>
    <vType id="BUS" probability="0.007955"/>
    <vType id="truck_trailer" probability="0.006048"/>
    <vType id="car_trailer" probability="0.004666"/>
    <vType id="semitrailer" probability="0.003756"/>
</vTypeDistribution>
